In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from joblib import Parallel, delayed
from collections import Counter
from typing import List, Union, Sequence, Optional, Dict
from customkernels import Kernel1Full

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score

import matplotlib.colors as mcolors

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from ucimlrepo import fetch_ucirepo 


In [ ]:
def subBagSVM_linear(X, y, m, no_svms=100, C_values=None, replace=True, n_jobs=-1, random_state=None):

    if C_values is None:
        C_values = 2.0 ** np.arange(-5, 6)
    
    C_values = np.asarray(C_values).ravel()

    rng = np.random.RandomState(random_state)
    X = np.asarray(X)
    y = np.asarray(y)
    n_samples, n_features = X.shape

    # Method fit one SVM with bootstrap + feature subspace
    def fit_one(seed):
        local_rng = np.random.RandomState(seed)
        idx_samples = local_rng.choice(n_samples, size=n_samples, replace=replace)
        # sample features
        feat_idx = local_rng.choice(n_features, size=m, replace=False)
        X_sub = X[idx_samples[:, None], feat_idx]  # shape (n_samples, m)
        y_sub = y[idx_samples]
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', GridSearchCV(
                SVC(kernel='linear', probability=False, random_state=seed),
                 param_grid={'C': C_values,
                             'class_weight':[ 'balanced',{0:1, 1:2},{0:1, 1:4}
                            ]},
                cv=3,
                scoring='accuracy',
                n_jobs=1
            ))
        ])
        pipe.fit(X_sub, y_sub)
        best_acc = pipe.named_steps['svc'].best_score_
        return pipe, best_acc, feat_idx

    # run in parallel
    results = Parallel(n_jobs=n_jobs)(
        delayed(fit_one)(seed)
        for seed in range(no_svms)
    )

    models, accuracies, feature_idx = zip(*results)
    ensemble = {
        'models': list(models),
        'accuracies': np.array(accuracies),
        'feature_idx': list(feature_idx)
    }
    return ensemble

def subBagSVM_rbf(X, y, m, no_svms=100, C_values=None, replace=True, n_jobs=-1, random_state=None):
    if C_values is None:
        C_values = 2.0 ** np.arange(-5, 6)
    
    C_values = np.asarray(C_values).ravel()

    rng = np.random.RandomState(random_state)
    X = np.asarray(X)
    y = np.asarray(y)
    n_samples, n_features = X.shape

    # Method fit one SVM with bootstrap + feature subspace
    def fit_one(seed):
        local_rng = np.random.RandomState(seed)
        idx_samples = local_rng.choice(n_samples, size=n_samples, replace=replace)
        # sample features
        feat_idx = local_rng.choice(n_features, size=m, replace=False)
        X_sub = X[idx_samples[:, None], feat_idx]  # shape (n_samples, m)
        y_sub = y[idx_samples]
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', GridSearchCV(
                SVC(kernel='rbf', probability=False, random_state=seed),
                 param_grid={'C': C_values,
                             'class_weight':[ 'balanced',{0:1, 1:2},{0:1, 1:4}
                            ]},
                cv=3,
                scoring='accuracy',
                n_jobs=1
            ))
        ])
        pipe.fit(X_sub, y_sub)
        best_acc = pipe.named_steps['svc'].best_score_
        return pipe, best_acc, feat_idx

    # run in parallel
    results = Parallel(n_jobs=n_jobs)(
        delayed(fit_one)(seed)
        for seed in range(no_svms)
    )

    models, accuracies, feature_idx = zip(*results)
    ensemble = {
        'models': list(models),
        'accuracies': np.array(accuracies),
        'feature_idx': list(feature_idx)
    }
    return ensemble


def predict_subBagSVM(ensemble, X_new):
    X_new = np.asarray(X_new)
    n_models = len(ensemble['models'])
    n_samples_new = X_new.shape[0]
    # collect each model's predictions
    votes = np.empty((n_samples_new, n_models), dtype=object)
    for i, (pipe, feat_idx) in enumerate(zip(ensemble['models'], ensemble['feature_idx'])):
        X_sub = X_new[:, feat_idx]
        votes[:, i] = pipe.predict(X_sub)

    # majority vote
    def majority_label(row_votes):
        vals, counts = np.unique(row_votes, return_counts=True)
        return vals[np.argmax(counts)]

    preds = np.apply_along_axis(majority_label, axis=1, arr=votes)
    return preds

In [ ]:
if __name__ == "__main__":

    # uncomment each section according to dataset being used
    # 1. dmean dataset
    # df = pd.read_csv("dmean_df.csv")
    # y = df["ExtractionFlag"]
    # X = df.drop(columns=["ExtractionFlag"])

    # # 2. dmax dataset
    # # df = pd.read_csv("dmax_df.csv")
    # # y = df["ExtractionFlag"]
    # # X = df.drop(columns=["ExtractionFlag"])

    # cont_cols = ["TotalDose"]   
    # cat_cols = X.drop(columns=['TotalDose']).columns.tolist()

    # X_dummies = pd.get_dummies(
    #     X,
    #     columns=[c for c in X.columns if c not in ["TotalDose", "PatientID"]],
    #     drop_first=False,
    #     dtype=float
    # )

    # #  this splitting procedure is only for extraction datasets
    # unique_patients = df['PatientID'].unique()

    # train_patients, test_patients = train_test_split(
    #     unique_patients, 
    #     test_size=0.2, 
    #     random_state=42
    # )

    # train_mask = df['PatientID'].isin(train_patients)
    # test_mask = df['PatientID'].isin(test_patients)

    # X_train = X_dummies[train_mask]
    # X_test = X_dummies[test_mask]
    # y_train = y[train_mask]
    # y_test = y[test_mask]

    # X_train = X_train.drop(columns=["PatientID"])
    # X_test = X_test.drop(columns=["PatientID"])

    # print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

    # df 3: congressional_voting_records
    congressional_voting_records = fetch_ucirepo(id=105) 
    X = congressional_voting_records.data.features 
    y = congressional_voting_records.data.targets 
    y = y.iloc[:, 0] 
    y = y.map({'republican': 0, 'democrat': 1})
    cat_cols = X.columns.tolist()   
    cont_cols = []  # No continuous columns in this dataset

    # df 4: MONK dataset
    # monk_s_problems = fetch_ucirepo(id=70) 
    # X = monk_s_problems.data.features 
    # y = monk_s_problems.data.targets 
    # y = y.map({'republican': 0, 'democrat': 1})
    # cat_cols = X.columns.tolist()   
    # cont_cols = []  # No continuous columns in this dataset

    # this split is used for the UCI datasets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
        )
    print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

    # m_values = [30, 40, 50, 60, 70, 80,  88] # for extrcation datasets
    m_values = [10, 18, 25, 28, 32] # congressional voting datasets
    # m_values = [10, 15, 30, 35, 39] # Monk dataset
    no_svms = 50  
    C_values = [0.1, 1, 10]
    random_state = 42  

    # Store results
    results_linear_rsvm = []
    results_linear_rsm = []
    results_rbf_rsvm = []
    results_rbf_rsm = []


    for m in m_values:
        print(f"Training m={m} ...")
        # ----- LINEAR RSVM -----
        ens_lin_rsvm = subBagSVM_linear(X_train.values, y_train.values, m=m, no_svms=no_svms,
                                        C_values=C_values, replace=True, n_jobs=-1, random_state=random_state)
        accs_lin_rsvm = []
        for i in range(1, no_svms + 1):
            sub_ens = {'models': ens_lin_rsvm['models'][:i], 'feature_idx': ens_lin_rsvm['feature_idx'][:i]}
            pred = predict_subBagSVM(sub_ens, X_test.values)
            accs_lin_rsvm.append(accuracy_score(y_test, pred))
        results_linear_rsvm.append(accs_lin_rsvm)
        print("done training linear RSVM")

        # ----- LINEAR RSM -----
        ens_lin_rsm = subBagSVM_linear(X_train.values, y_train.values, m=m, no_svms=no_svms,
                                    C_values=C_values, replace=False, n_jobs=-1, random_state=random_state)
        accs_lin_rsm = []
        for i in range(1, no_svms + 1):
            sub_ens = {'models': ens_lin_rsm['models'][:i], 'feature_idx': ens_lin_rsm['feature_idx'][:i]}
            pred = predict_subBagSVM(sub_ens, X_test.values)
            accs_lin_rsm.append(accuracy_score(y_test, pred))
        results_linear_rsm.append(accs_lin_rsm)
        print("done training linear RSM")

        # ----- RBF RSVM -----
        ens_rbf_rsvm = subBagSVM_rbf(X_train.values, y_train.values, m=m, no_svms=no_svms,
                                    C_values=C_values, replace=True, n_jobs=-1, random_state=random_state)
        accs_rbf_rsvm = []
        for i in range(1, no_svms + 1):
            sub_ens = {'models': ens_rbf_rsvm['models'][:i], 'feature_idx': ens_rbf_rsvm['feature_idx'][:i]}
            pred = predict_subBagSVM(sub_ens, X_test.values)
            accs_rbf_rsvm.append(accuracy_score(y_test, pred))
        results_rbf_rsvm.append(accs_rbf_rsvm)
        print("done training RBF RSVM")

        # ----- RBF RSM -----
        ens_rbf_rsm = subBagSVM_rbf(X_train.values, y_train.values, m=m, no_svms=no_svms,
                                    C_values=C_values, replace=False, n_jobs=-1, random_state=random_state)
        accs_rbf_rsm = []
        for i in range(1, no_svms + 1):
            sub_ens = {'models': ens_rbf_rsm['models'][:i], 'feature_idx': ens_rbf_rsm['feature_idx'][:i]}
            pred = predict_subBagSVM(sub_ens, X_test.values)
            accs_rbf_rsm.append(accuracy_score(y_test, pred))
        results_rbf_rsm.append(accs_rbf_rsm)
        print("done training RBF RSM")



colors = plt.cm.tab10.colors 

fig, axs = plt.subplots(1, 2, figsize=(14, 6.5), sharey=True)

# ----- Plotting -----
for i, (rsvm, rsm) in enumerate(zip(results_linear_rsvm, results_linear_rsm)):
    color = colors[i % len(colors)]
    axs[0].plot(range(1, no_svms + 1), rsvm, color=color, linestyle='-', linewidth=2)
    axs[0].plot(range(1, no_svms + 1), rsm, color=color, linestyle='--', linewidth=2)

axs[0].set_title("Linear Kernel")
axs[0].set_xlabel("Number of SVMs")
axs[0].set_ylabel("Test Accuracy")

for i, (rsvm, rsm) in enumerate(zip(results_rbf_rsvm, results_rbf_rsm)):
    color = colors[i % len(colors)]
    axs[1].plot(range(1, no_svms + 1), rsvm, color=color, linestyle='-', linewidth=2)
    axs[1].plot(range(1, no_svms + 1), rsm, color=color, linestyle='--', linewidth=2)

axs[1].set_title("RBF Kernel")
axs[1].set_xlabel("Number of SVMs")

# ----- Legend Handles -----
color_handles = [
    mlines.Line2D([], [], color=colors[i % len(colors)], linestyle='-', linewidth=3, label=f'{m_values[i]}')
    for i in range(len(m_values))
]

style_handles = [
    mlines.Line2D([], [], color='black', linestyle='--', linewidth=2, label='RSM'),
    mlines.Line2D([], [], color='black', linestyle='-', linewidth=2, label='RSVM')
]

all_handles = color_handles + style_handles
all_labels = [h.get_label() for h in all_handles]

fig.legend(
    handles=all_handles,
    labels=all_labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=len(all_handles),
    frameon=False
)

plt.tight_layout(rect=[0, 0.15, 1, 1])
plt.show()